In [ ]:
import polars as pl

pdb_v = pl.read_parquet(
    "../../data/pdb/triad/staged/pdb_validation_triad.conf_af3.parquet"
)

In [2]:
import numpy as np


def spearman_corr_matrix(df: pl.DataFrame, columns) -> pl.DataFrame:

    # rank-transform columns
    ranked = df.select([pl.col(c).rank("average").alias(c) for c in columns])
    # Pearson on ranks = Spearman
    corr = np.corrcoef(ranked.to_numpy(), rowvar=False)
    # build a labeled square matrix in Polars
    mat = pl.DataFrame({c: corr[:, i] for i, c in enumerate(columns)})
    mat = mat.with_columns(var1=pl.Series(columns)).select("var1", *columns)
    return mat

In [11]:
interface_feats = [
    "mean_p_tcr_interface_pae",
    "mean_tcr_pmhc_interface_pae",
    "mean_p_tcr_interface_contact_prob",
    "mean_tcr_pmhc_interface_contact_prob",
    "mean_p_tcr_pae",
    "mean_tcr_p_pae",
    "mean_mhc_tcr_pae",
    "mean_tcr_mhc_pae",
    "mean_p_mhc_pae",
    "tcr_mhc_contacts",
    "peptide_tcr_contacts",
]


local_feats = [
    "peptide_mean_pLDDT",
    "tcr_1_cdr_1_mean_pLDDT",
    "tcr_1_cdr_2_mean_pLDDT",
    "tcr_1_cdr_2_5_mean_pLDDT",
    "tcr_1_cdr_3_mean_pLDDT",
    "tcr_2_cdr_1_mean_pLDDT",
    "tcr_2_cdr_2_mean_pLDDT",
    "tcr_2_cdr_2_5_mean_pLDDT",
    "tcr_2_cdr_3_mean_pLDDT",
    "tcr_cdrs_mean_pLDDT",
    "mhc_helices_mean_pLDDT",
]

summary_feats = [
    "iptm",
    "ptm",
    "ranking_score",
]

all_feats = interface_feats + local_feats + summary_feats

df = spearman_corr_matrix(pdb_v, all_feats)

# df = df.filter([pl.col(colname).abs() < 0.5 for colname in df.columns if colname !="var1"])

In [15]:
df.filter([pl.col(colname).abs() < 1 for colname in df.columns if colname != "var1"])

var1,mean_p_tcr_interface_pae,mean_tcr_pmhc_interface_pae,mean_p_tcr_interface_contact_prob,mean_tcr_pmhc_interface_contact_prob,mean_p_tcr_pae,mean_tcr_p_pae,mean_mhc_tcr_pae,mean_tcr_mhc_pae,mean_p_mhc_pae,tcr_mhc_contacts,peptide_tcr_contacts,peptide_mean_pLDDT,tcr_1_cdr_1_mean_pLDDT,tcr_1_cdr_2_mean_pLDDT,tcr_1_cdr_2_5_mean_pLDDT,tcr_1_cdr_3_mean_pLDDT,tcr_2_cdr_1_mean_pLDDT,tcr_2_cdr_2_mean_pLDDT,tcr_2_cdr_2_5_mean_pLDDT,tcr_2_cdr_3_mean_pLDDT,tcr_cdrs_mean_pLDDT,mhc_helices_mean_pLDDT,iptm,ptm,ranking_score
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""peptide_tcr_contacts""",-0.899095,-0.906709,0.633023,0.735495,-0.879741,-0.900723,-0.899023,-0.893321,-0.35982,0.64304,1.0,0.726158,0.621817,0.768288,0.59488,0.603991,0.735167,0.798066,0.654611,0.748459,0.835523,0.813608,0.898759,0.889556,0.90047
"""ptm""",-0.955147,-0.976079,0.536481,0.699212,-0.943628,-0.970067,-0.978061,-0.98886,-0.441284,0.703672,0.889556,0.816136,0.738844,0.81232,0.721151,0.681616,0.845367,0.917281,0.760488,0.795863,0.926893,0.909296,0.99456,1.0,0.974593
"""ranking_score""",-0.966746,-0.974001,0.529745,0.681098,-0.951033,-0.975608,-0.983393,-0.983254,-0.437352,0.710475,0.90047,0.803044,0.720003,0.825847,0.665374,0.627597,0.794714,0.872908,0.706815,0.775786,0.885924,0.89067,0.982545,0.974593,1.0
